In [ ]:
import pandas as pd
import numpy as np

import seaborn as sns
import matplotlib.pyplot as plt

from pathlib import Path

In [ ]:
FOLDER = Path("../../experiments/lfm/outputs").resolve()
N_EPOCHS = 20
baseline = f"baseline-torch-qat-int8b-{N_EPOCHS}e"


def read_data(run):
    df = pd.read_csv(FOLDER / run / "metrics.csv")
    for col in [
        "epoch",
        "step",
        "t",
        "train_loss",
        "val_loss",
        "val_perplexity",
        "quant_error",
    ]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")
    if "val_perplexity" not in df.columns and "val_loss" in df.columns:
        df["val_perplexity"] = np.exp(df["val_loss"].clip(upper=20))
    return df.sort_values("epoch").reset_index(drop=True)

In [ ]:
def compare_runs(run, baseline, metrics=None):
    if metrics is None:
        metrics = [
            ("train_loss", "Train loss"),
            ("val_loss", "Val loss"),
            ("val_perplexity", "Val perplexity"),
            ("t", "Quantization t"),
            ("quant_error", "Quant error"),
        ]

    run_df = read_data(run)
    base_df = read_data(baseline)

    fig, axes = plt.subplots(2, 3, figsize=(14, 8))
    for ax, (col, title) in zip(axes.flatten(), metrics):
        if col not in run_df.columns:
            ax.set_title(f"{title} (missing)")
            ax.axis("off")
            continue
        if run_df[col].notna().any():
            sns.lineplot(run_df, x="epoch", y=col, ax=ax, label=run)
        if col in base_df.columns and base_df[col].notna().any():
            sns.lineplot(base_df, x="epoch", y=col, ax=ax, label=baseline)
        ax.set_title(title)
        ax.grid(True)
        ax.legend()

    for ax in axes.flatten()[len(metrics):]:
        ax.axis("off")

    plt.tight_layout()
    plt.show()

In [ ]:
compare_runs(
    "cos-0.9-8b-20e-wd0.0",
    baseline,
)

In [ ]:
def compare_perplexity(run, baseline):
    run_df = read_data(run)
    base_df = read_data(baseline)

    fig, ax = plt.subplots(figsize=(8, 5))

    if "val_perplexity" in run_df.columns and run_df["val_perplexity"].notna().any():
        sns.lineplot(run_df, x="epoch", y="val_perplexity", ax=ax, label=run)
    if "val_perplexity" in base_df.columns and base_df["val_perplexity"].notna().any():
        sns.lineplot(base_df, x="epoch", y="val_perplexity", ax=ax, label=baseline)

    ax.set_yscale("log")
    ax.set_xlabel("epoch")
    ax.set_ylabel("val_perplexity (log scale)")
    ax.grid(True, which="both", ls="--", alpha=0.4)
    ax.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
compare_perplexity(
    "cos-0.9-8b-20e-wd0.0",
    baseline,
)